<img align="left" src="https://docs.linea.org.br/images/logo-linea.fw_2.png" width=100 style="padding: 40px">  
<img align="left" src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width=180>

<br clear="left">

# Photo-z HATS catalog QA

This report characterizes the output pz-compute HATS catalog using scalar columns only. It reports catalog size, schema, descriptive statistics, and redshift distributions for the complete catalog and optional filtered samples.

## 1. Configuration

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import HTML, Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'pz_qa.py').exists():
    NOTEBOOK_DIR = Path('doc/notebooks/pz_compute_v2').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from pz_qa import (
    columns_box, execution_metadata_html, load_config,
    plot_redshift_distributions, plot_spectroscopic_comparison, run_qa,
    selection_report, start_cluster, write_report,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda value: f'{value:,.6g}')

config_value = os.environ.get('PZ_QA_CONFIG')
if not config_value:
    raise RuntimeError('Set PZ_QA_CONFIG before starting Jupyter.')
CONFIG_PATH = Path(config_value).expanduser().resolve()
config = load_config(CONFIG_PATH)
display(HTML(execution_metadata_html(config)))
display(pd.Series(config['run'], name='value').rename_axis('property').to_frame())
display(Markdown('### Configured redshift samples'))
display(selection_report(config).style.set_properties(**{'white-space': 'normal', 'text-align': 'left'}))

## 2. Start Dask

Cluster settings come from the selected YAML file. This QA projects scalar columns only.

In [ ]:
client, cluster = start_cluster(config)
display(client)
display(pd.Series(config['cluster'], name='value').rename_axis('setting').to_frame())

## 3. Scan scalar HATS columns

In [ ]:
result = run_qa(config)
display(Markdown('Scalar-only scan completed.'))

## 4. Catalog overview

In [ ]:
overview = pd.Series({
    'rows': result.row_count,
    'top-level columns': len(result.columns),
    'scalar columns': len(result.columns) - len(result.nested_columns),
    'nested columns not read': len(result.nested_columns),
    'HATS partitions': result.partition_count,
    'redshift column': config['redshift']['column'],
}, name='value')
display(overview.rename_axis('property').to_frame())

## 5. Available columns

The scrollable box lists every top-level HATS column. Nested columns are visible for catalog characterization but were not read.

In [ ]:
display(HTML(columns_box(result)))
display(Markdown('### Schema'))
display(result.schema)

## 6. Basic scalar-column statistics

This is a distributed `describe` of every selected numeric scalar column. The statistics-only projection uses `float64` so nullable numeric dtypes are supported and unsigned integer arithmetic does not overflow; original dtypes remain visible in the schema table. `null` is derived from the HATS row count minus the non-null count reported by `describe`. Nested and non-numeric columns are omitted.

In [ ]:
display(result.describe)

## 7. Global and filtered sample sizes

In [ ]:
display(result.selection_counts.style.format({'retained fraction': '{:.3%}'}))

## 8. Redshift distributions

The left panel shows object counts. The right panel normalizes each configured sample independently, making shape comparisons possible when filtered samples have different sizes.

In [ ]:
figure, axes = plot_redshift_distributions(result)
plt.show()

In [ ]:
if result.spectroscopic_metrics is not None:
    display(Markdown('### Comparison with spectroscopy'))
    display(Markdown(
        'Bias and NMAD use the residual `(z_phot - z_spec) / (1 + z_spec)`. '
        'Outliers have an absolute residual above 0.15.'
    ))
    display(result.spectroscopic_metrics.style.format({'outlier fraction': '{:.2%}'}, na_rep='—'))
    figure, axes = plot_spectroscopic_comparison(result)
    plt.show()

## 9. Optional compact export

In [ ]:
report_dir = write_report(result)
print('Report tables were not written (paths.report_dir is null).' if report_dir is None else f'Report tables: {report_dir}')

## 10. Close Dask

In [ ]:
client.close()
if cluster is not None:
    cluster.close()

### Acknowledgments

*This notebook was supported by resources supplied by the Laboratório Interinstitucional de e-Astronomia (LIneA).*  
*These results were obtained using RAIL, an open-source software package made available by the LSST Dark Energy Science Collaboration (DESC).*